## 1. Imports and Data

In [204]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf


In [205]:

tickers = {
    "Information Technology": ["AAPL", "MSFT", "NVDA", "AVGO", "CRM"],  # High momentum, growth factor exposure
    "Financials": ["JPM", "BAC", "GS", "MS", "BLK", "WFC"],             # Value factor, rate sensitivity
    "Health Care": ["JNJ", "UNH", "LLY", "ABBV", "MRK", "PFE"],         # Defensive, quality factor
    "Consumer Discretionary": ["AMZN", "TSLA", "HD", "MCD", "NKE", "LOW"], # Cyclical, momentum variation
    "Industrials": ["CAT", "HON", "UNP", "RTX", "GE", "DE"],            # Classic value/quality mix
    "Communication Services": ["GOOGL", "META", "DIS", "NFLX", "T"],    # Growth vs. value spread
    "Consumer Staples": ["PG", "KO", "PEP", "WMT", "COST", "CL"],       # Low vol, defensive
    "Energy": ["XOM", "CVX", "COP", "SLB", "EOG"],                      # Value, commodity beta
    "Utilities": ["NEE", "DUK", "SO", "AEP", "EXC"],                    # Low vol, yield factor
    "Real Estate": ["PLD", "AMT", "EQIX", "SPG", "PSA"],                # Yield, rate sensitivity
    "Materials": ["LIN", "APD", "NEM", "FCX", "SHW"],                   # Cyclical, commodity exposure
}

all_tickers = [ticker for sector in tickers.values() for ticker in sector]



In [206]:
start_date = "2010-01-01"
end_date = "2026-08-01"

data = yf.download(
    [ticker for sector in tickers.values() for ticker in sector],
    start=start_date,
    end=end_date,
    interval = "1mo"
)["Close"]

# Compute returns
returns = np.log(data).diff().dropna()

# Data Quality Checks
# 1. Are there any NaNs?
print(returns.isna().sum().sum())

# 2. Are they at the start?
print(returns.iloc[0].isna().sum())

# 3. Flag outliers (returns > 50% or < -50%)
outliers = (returns > 0.5) | (returns < -0.5)
outliers_df = returns[outliers].stack(level = 'Ticker')
print(outliers_df)


C:\Users\godwi\AppData\Local\Temp\ipykernel_19688\3596477166.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[                       0%                       ]

[*********************100%***********************]  60 of 60 completed


0
0
Date        Ticker
2013-05-01  TSLA      0.593717
2016-02-01  FCX       0.506032
2020-03-01  EOG      -0.565959
            SLB      -0.697216
            SPG      -0.808050
2020-08-01  TSLA      0.554719
2022-04-01  NFLX     -0.676915
dtype: float64


In [207]:
# Compare to benchmark
spy_row = yf.download(
    "SPY",
    start=start_date,
    end=end_date,
    interval = "1mo"
)

spy_prices = spy_row["Close"]["SPY"]

spy_returns = np.log(spy_prices).diff().dropna()
spy_returns = spy_returns.reindex(returns.index)
#spy_returns.to_csv("data/spy_returns.csv")

# Check shape
print(returns.shape)        # (T, N)
print(spy_returns.shape)    # (T,)
# Confirm all share the same index
assert returns.index.equals(spy_returns.index)

assert len(returns) == len(spy_returns), \
    "returns and spy_returns have different lengths"


C:\Users\godwi\AppData\Local\Temp\ipykernel_19688\3775808830.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_row = yf.download(
[*********************100%***********************]  1 of 1 completed

(162, 60)
(162,)


## 2. Raw Signals

In [208]:
# Pull Fundamental Data
# Market Cap
shares_dict = {}
for ticker in all_tickers:
    try:
        info = yf.Ticker(ticker).info
        shares = info.get('sharesOutstanding', None)
        shares_dict[ticker] = shares
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        shares_dict[ticker] = None

shares_series = pd.Series(shares_dict)
missing = shares_series[shares_series.isna()]
print(missing) # None
market_cap = data.multiply(shares_series, axis=1)
#market_cap.to_csv("data/market_cap.csv")
log_market_cap = np.log(market_cap)
#log_market_cap.to_csv("data/log_market_cap.csv")

print(log_market_cap.shape) # 120 Months x 60 stocks
print(log_market_cap.head())

Series([], dtype: int64)
(199, 60)
                 AAPL  ABBV        AEP        AMT       AMZN        APD  \
Date                                                                      
2010-01-01  25.159348   NaN  23.028371  23.375253  24.934689  23.058531   
2010-02-01  25.222694   NaN  23.010457  23.380188  24.877169  22.956325   
2010-03-01  25.361126   NaN  23.026977  23.379015  25.014063  23.038330   
2010-04-01  25.466406   NaN  23.030482  23.335853  25.023811  23.075885   
2010-05-01  25.450149   NaN  22.972487  23.328969  24.935087  22.969917   

                 AVGO        BAC        BLK        CAT  ...        SLB  \
Date                                                    ...              
2010-01-01  22.481703  25.143106  23.808491  23.507536  ...  24.887843   
2010-02-01  22.525053  25.236137  23.831514  23.595615  ...  24.853193   
2010-03-01  22.649730  25.305738  23.831388  23.692438  ...  24.891091   
2010-04-01  22.647295  25.304617  23.662931  23.778736  ...  25.00925

In [209]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
# Compute Raw Signal Values
# 1. Market Beta
# Run OLS over past 36 months
beta_window = 36
monthly_index = returns.index
beta_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

assert returns.index.equals(spy_returns.index), \
    "returns and spy_returns have different indices — align them first"

for i, date in enumerate(monthly_index):
    if i < beta_window:
        continue  # Not enough data for the first few months


    window_returns = returns.iloc[i-beta_window:i]
    window_spy = spy_returns.iloc[i-beta_window:i]
    
    valid_mask = window_spy.notna() 
    if valid_mask.sum() < 24:
        continue

    spy_window_clean = window_spy[valid_mask] 

    for ticker in all_tickers:
        y = window_returns.loc[window_spy.index, ticker]

        combined_valid = valid_mask & y.notna()

        if combined_valid.sum() < 24: 
            continue

        y_clean = y[combined_valid].values
        spy_clean = window_spy[combined_valid].values
        X_clean = add_constant(spy_clean, has_constant="add")  # Recreate X with cleaned spy data

        try:
            model = OLS(y_clean, X_clean).fit()
            beta_panel.loc[date, ticker] = model.params[1]  # Store the beta coefficient

        except Exception:
            pass

#beta_panel.to_csv("data/signal_beta_raw.csv")


    

In [210]:
# Sanity check
print((beta_panel < -1).sum().sum())
print((beta_panel > 3).sum().sum()) 
beta_panel.describe()

0
12


,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
count,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,...,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000,126.000000
mean,1.209636,1.022453,1.805490,1.020934,1.244416,1.162688,1.452278,1.362714,1.409344,1.417206,...,1.055099,0.665634,0.698320,1.253697,0.468783,0.851715,0.865909,0.264067,2.106497,1.224097
std,0.147872,0.180441,0.510038,0.232671,0.182417,0.088932,0.179338,0.130613,0.196578,0.195939,...,0.274595,0.344080,0.296607,0.592673,0.383407,0.229597,0.171139,0.251337,0.573032,0.153389
min,0.816837,0.749327,0.923619,0.298529,0.870038,0.933788,1.156367,0.993031,0.939033,1.067051,...,0.691076,-0.034940,0.150481,0.418338,-0.147801,0.372992,0.354440,-0.401136,0.880385,0.949856
25%,1.119135,0.881333,1.396079,0.936142,1.112423,1.102637,1.265336,1.283272,1.265010,1.235443,...,0.850502,0.277095,0.478719,0.611106,0.145217,0.682722,0.748533,0.187825,1.710685,1.076048
50%,1.218818,1.010644,1.621059,1.026680,1.229381,1.178622,1.462306,1.364371,1.478110,1.431638,...,0.959851,0.812275,0.674263,1.377599,0.378051,0.899226,0.837896,0.322263,2.112223,1.220997
75%,1.282854,1.078141,2.318634,1.162246,1.378439,1.226579,1.608653,1.459681,1.570675,1.583783,...,1.268017,0.936230,0.936550,1.851083,0.908125,1.059127,0.972549,0.430951,2.458301,1.352224
max,1.537399,1.452953,2.727059,1.615778,1.602710,1.421822,1.801078,1.669258,1.725443,1.715845,...,1.733340,1.129349,1.233570,2.233652,1.132596,1.172600,1.301418,0.736314,3.260807,1.563773


In [211]:
"""
fig, axes = plt.subplots(10, 6, figsize=(18, 28))
axes = axes.flatten()

for i, ticker in enumerate(beta_panel.columns):
    ax = axes[i]
    beta_panel[ticker].plot(ax=ax, linewidth=1.5)
    mean_beta = beta_panel[ticker].mean()
    ax.axhline(mean_beta, color="red", ls="--", linewidth=1, label="Mean")
    ax.set_title(f"{ticker}, Mean Beta = {mean_beta:.2f}")
    ax.set_ylabel("Beta")

for j in range(len(beta_panel.columns), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Rolling 36-Month Beta vs SPY", y=1.002)
fig.tight_layout()
plt.savefig("outputs/beta_panel.png")
plt.show()
"""

'\nfig, axes = plt.subplots(10, 6, figsize=(18, 28))\naxes = axes.flatten()\n\nfor i, ticker in enumerate(beta_panel.columns):\n    ax = axes[i]\n    beta_panel[ticker].plot(ax=ax, linewidth=1.5)\n    mean_beta = beta_panel[ticker].mean()\n    ax.axhline(mean_beta, color="red", ls="--", linewidth=1, label="Mean")\n    ax.set_title(f"{ticker}, Mean Beta = {mean_beta:.2f}")\n    ax.set_ylabel("Beta")\n\nfor j in range(len(beta_panel.columns), len(axes)):\n    fig.delaxes(axes[j])\n\nfig.suptitle("Rolling 36-Month Beta vs SPY", y=1.002)\nfig.tight_layout()\nplt.savefig("outputs/beta_panel.png")\nplt.show()\n'

In [212]:
# 2. Momentum
mom_start = 12 # 12 months lookback
mom_end = 1 # exclude most recent month - noisy
momentum_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < mom_start:
        continue  # Not enough data for the first few months

    window = returns.iloc[i-mom_start:i-mom_end] 
    compounded = (1 + window).prod(axis = 0, skipna = False) - 1
    momentum_panel.loc[date] = compounded

#momentum_panel.to_csv("data/signal_momentum_raw.csv")
print(momentum_panel)

                AAPL      MSFT      NVDA      AVGO       CRM       JPM  \
Date                                                                     
2013-02-01       NaN       NaN       NaN       NaN       NaN       NaN   
2013-03-01       NaN       NaN       NaN       NaN       NaN       NaN   
2013-04-01       NaN       NaN       NaN       NaN       NaN       NaN   
2013-05-01       NaN       NaN       NaN       NaN       NaN       NaN   
2013-06-01       NaN       NaN       NaN       NaN       NaN       NaN   
...              ...       ...       ...       ...       ...       ...   
2026-03-01  0.054916  0.058279  0.432182  0.536899 -0.325148  0.164223   
2026-04-01  0.174706  0.021190  0.542687  0.788864 -0.311413  0.236644   
2026-05-01  0.179865 -0.086373  0.510778  0.523209 -0.342622  0.207036   
2026-06-01  0.332167 -0.131911  0.410347  0.609748 -0.369813  0.197700   
2026-07-01  0.487948 -0.112726  0.288374  0.518290 -0.338873  0.045385   

                 BAC        GS       

In [213]:
# 3. Size
size_panel = log_market_cap.copy()
size_panel = size_panel.reindex(index=monthly_index)
size_panel = size_panel.reindex(columns=all_tickers)

assert size_panel.index.equals(monthly_index), \
    "Index mismatch between size panel and returns index"

assert size_panel.columns.tolist() == all_tickers, \
    "Column mismatch between size panel and returns columns"

#size_panel.to_csv("data/signal_size_raw.csv")

In [214]:
# 4. Low Volatility
vol_window = 12
low_vol_panel = -returns.rolling(window = vol_window, min_periods = 10).std() # Negative for low vol = high score
low_vol_panel = low_vol_panel.reindex(index=monthly_index)
low_vol_panel = low_vol_panel.reindex(columns=all_tickers)
#low_vol_panel.to_csv("data/signal_lowvol_raw.csv")

In [215]:
# 5. Long-term Reversal
ltr_start = 36
ltr_end = 12
ltr_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < ltr_start:
        continue  
    
    window = returns.iloc[i-ltr_start:i-ltr_end] 
    compounded = (1 + window).prod(axis = 0, skipna = False) - 1
    ltr_panel.loc[date] = -compounded 

#ltr_panel.to_csv("data/signal_ltr_raw.csv")

In [216]:
'''
dates = np.load("data/dates.npy", allow_pickle=True) 
tickers = np.load("data/tickers.npy", allow_pickle=True)  
factors = np.load("data/factors.npy", allow_pickle=True) 


data = pd.read_csv("data/prices.csv", index_col=0, parse_dates=True)
returns = pd.read_csv("data/returns.csv", index_col=0, parse_dates=True).reindex(dates)
spy_returns = pd.read_csv("data/spy_returns.csv", index_col=0, parse_dates=True).reindex(dates)
mktcap_panel = pd.read_csv("data/market_cap.csv", index_col=0, parse_dates=True).reindex(dates)
beta_panel = pd.read_csv("data/signal_beta_raw.csv", index_col=0, parse_dates=True)
momentum_panel = pd.read_csv("data/signal_momentum_raw.csv", index_col=0, parse_dates=True)
size_panel = pd.read_csv("data/signal_size_raw.csv", index_col=0, parse_dates=True)
low_vol_panel = pd.read_csv("data/signal_lowvol_raw.csv", index_col=0, parse_dates=True)
ltr_panel = pd.read_csv("data/signal_ltr_raw.csv", index_col=0, parse_dates=True)
'''



print(beta_panel.shape)
print(momentum_panel.shape)
print(size_panel.shape)
print(low_vol_panel.shape)
print(ltr_panel.shape)

(162, 60)
(162, 60)
(162, 60)
(162, 60)
(162, 60)


In [217]:
beta_panel.tail()

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-01,0.895633,1.148674,2.127046,1.201818,1.276638,1.195227,1.416525,1.382583,1.503590,1.393849,...,1.606176,0.713018,0.946208,1.304084,1.026582,0.588942,0.740186,0.196887,0.960315,1.492747
2026-04-01,0.848415,1.082688,1.973900,1.191310,1.125962,1.214759,1.430975,1.361881,1.470120,1.453150,...,1.565419,0.798485,0.844469,1.381008,1.107613,0.578276,0.585843,0.460835,1.132484,1.563773
2026-05-01,0.816837,1.076418,1.825464,1.452233,0.870038,1.109888,1.348761,1.262926,1.479785,1.393848,...,1.449975,0.798921,0.873078,1.296218,1.132596,0.493722,0.561876,0.395305,0.919049,1.320091
2026-06-01,0.894653,1.132801,1.855185,1.484033,0.939554,1.027923,1.265247,1.266670,1.461379,1.333539,...,1.419508,0.771362,0.845625,1.234268,1.097554,0.458378,0.473907,0.318457,0.949523,1.236897
2026-07-01,0.897507,1.245748,1.901490,1.615778,1.121204,0.978401,1.239499,1.366861,1.502316,1.384894,...,1.506363,0.808357,0.852101,1.144370,1.068163,0.388662,0.354440,0.409735,0.880385,1.069002


In [218]:
# Winsorise, Z-score normalisation and Burn-in
# Set values < 1%ile to 1%ile, > 99%ile to 99%ile, across all stocks, at each time t - hence 'cross sectional'
def winsorise(signal_panel: pd.DataFrame, lower = 0.01, upper = 0.99) -> pd.DataFrame:
    def winsorise_row(row):
        lower_bound = row.quantile(lower)
        upper_bound = row.quantile(upper)
        return row.clip(lower=lower_bound, upper=upper_bound)
    
    return signal_panel.apply(winsorise_row, axis=1)

def standardise(signal_panel: pd.DataFrame) -> pd.DataFrame:
    def standardise_row(row):
        mu = row.mean()
        sig = row.std()
        if sig == 0:
            return row - mu
        return (row - mu) / sig

    return signal_panel.apply(standardise_row, axis=1)

def trim_burn_in(signal_panel: pd.DataFrame, burn_in_periods = 36) -> pd.DataFrame:
    return signal_panel.iloc[burn_in_periods:]

def normalise(df: pd.DataFrame) -> pd.DataFrame:
    def normalise_row(row):
        tot = row.abs().sum()
        return row / tot

    return df.apply(normalise_row, axis=1)



In [219]:
signals_raw = {
    "beta": beta_panel,
    "momentum": momentum_panel,
    "size": size_panel,
    "low_vol": low_vol_panel,
    "ltr": ltr_panel
}

signals_processed = {
    name: trim_burn_in(winsorise(standardise(panel))) for name, panel in signals_raw.items() 
}

#np.save("data/signals_raw.npy", signals_raw)
#np.save("data/signals_processed.npy", signals_processed)

In [220]:
# Trim signals due to burn-in
for signal in signals_processed.values():
    print(signal.isna().sum().sum())
    print(signal.shape)

# No NaNs


0
(126, 60)
0
(126, 60)
0
(126, 60)
0
(126, 60)
0
(126, 60)


In [221]:
signals_processed["beta"].tail()

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-01,0.118247,0.577897,2.134772,0.674433,0.810346,0.662461,1.064452,1.002796,1.222606,1.023260,...,1.408954,-0.213475,0.210116,0.860201,0.356116,-0.438862,-0.164124,-1.151032,0.235741,1.202910
2026-04-01,0.006820,0.431884,1.970048,0.628967,0.510400,0.671512,1.063813,0.938450,1.134838,1.104047,...,1.307747,-0.083771,-0.000339,0.973154,0.477108,-0.483316,-0.469587,-0.696401,0.522233,1.304761
2026-05-01,0.094317,0.567946,1.931199,1.253654,0.191388,0.629015,1.064859,0.908245,1.303924,1.147124,...,1.249534,0.061629,0.196934,0.968990,0.670448,-0.495234,-0.370880,-0.674804,0.280812,1.012548
2026-06-01,0.295040,0.722266,1.880136,1.352356,0.375591,0.534120,0.959866,0.962419,1.311716,1.082378,...,1.236601,0.073863,0.207086,0.904291,0.659033,-0.487614,-0.459755,-0.738625,0.393474,0.909007
2026-07-01,0.312107,0.927348,1.788039,1.581083,0.707314,0.455022,0.916308,1.141319,1.380629,1.173179,...,1.387778,0.154604,0.231887,0.748241,0.613605,-0.586877,-0.647337,-0.549646,0.281857,0.615089


## 3. Orthogonalise and weights via Information Coefficient

In [222]:
def residualise(target: pd.DataFrame, controls):
    if isinstance(controls, (pd.DataFrame, pd.Series)):
        controls = [controls]
    elif isinstance(controls, tuple):
        controls = list(controls)
    elif not isinstance(controls, list):
        raise TypeError("controls must be a DataFrame, Series, list, or tuple")

    controls = [
        ctrl.reindex(index=target.index, columns=target.columns)
        if isinstance(ctrl, pd.DataFrame)
        else ctrl.reindex(index=target.index)
        for ctrl in controls
    ]

    residuals = pd.DataFrame(index=target.index, columns=target.columns, dtype=float)

    for t in target.index:
        y = target.loc[t]
        X = pd.concat(
            [ctrl.loc[t].rename(f"ctrl_{i}") for i, ctrl in enumerate(controls)],
            axis=1,
        )

        common_idx = y.index.intersection(X.index)
        y = y.reindex(common_idx)
        X = X.reindex(common_idx)

        valid = y.notna() & X.notna().all(axis=1)
        if valid.sum() < 2:
            continue

        y_clean = y.loc[valid]
        X_clean = X.loc[valid]
        X_clean = add_constant(X_clean, has_constant="add")

        try:
            model = OLS(y_clean, X_clean).fit()
            residuals.loc[t, X_clean.index] = model.resid
        except Exception:
            continue

    return residuals

# 1. Size orthogonal to beta
size_clean = residualise(signals_processed["size"], signals_processed["beta"])

# 2. low vol orthogonal to beta + size
lowvol_clean = residualise(
    signals_processed["low_vol"],
    [signals_processed["beta"], size_clean],
)

# 3. momentum orthogonal to beta + size + low vol
momentum_clean = residualise(
    signals_processed["momentum"],
    [signals_processed["beta"], size_clean, lowvol_clean],
)

# 4. ltr orthogonal to beta + size + low vol + momentum
ltr_clean = residualise(
    signals_processed["ltr"],
    [signals_processed["beta"], size_clean, lowvol_clean, momentum_clean],
)

signals_cleaned = {
    "beta": signals_processed["beta"],
    "momentum": momentum_clean,
    "size": size_clean,
    "low_vol": lowvol_clean,
    "ltr": ltr_clean
}
    
#np.save("data/signals_cleaned.npy", signals_cleaned)

dates = signals_cleaned["beta"].index
factors = list(signals_cleaned.keys())




In [223]:
print(signals_cleaned["beta"].index)

DatetimeIndex(['2016-02-01', '2016-03-01', '2016-04-01', '2016-05-01',
               '2016-06-01', '2016-07-01', '2016-08-01', '2016-09-01',
               '2016-10-01', '2016-11-01',
               ...
               '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01',
               '2026-02-01', '2026-03-01', '2026-04-01', '2026-05-01',
               '2026-06-01', '2026-07-01'],
              dtype='datetime64[ns]', name='Date', length=126, freq='MS')


In [224]:
returns = returns.reindex(dates)

def compute_ic(name: str, signal: pd.DataFrame, returns: pd.DataFrame):
    returns = returns.shift(-1) # align r_{t+1}
    ic_series = pd.DataFrame(index = signal.index, columns = [name])
    
    for t in signal.index:
        x = signal.loc[t]
        r = returns.loc[t]

        valid = x.notna() & r.notna()

        ic = x[valid].corr(r[valid])
        ic_series.loc[t] = ic

    return ic_series



In [225]:
ic_df = pd.DataFrame(index = dates)
for name, signal in signals_processed.items():
    ic_df = pd.concat([ic_df, compute_ic(name, signal, returns)], axis = 1)
ic_df.dropna(inplace = True)
weights_rolling = standardise(ic_df).rolling(window = 24).mean().dropna()
weights_rolling

,beta,momentum,size,low_vol,ltr
Date,,,,,
2018-01-01,0.490243,0.238385,-0.001859,-0.406325,-0.320445
2018-02-01,0.402017,0.213572,-0.015049,-0.303879,-0.296661
2018-03-01,0.330785,0.247249,0.070685,-0.281978,-0.366740
2018-04-01,0.395211,0.236068,0.077270,-0.356393,-0.352156
2018-05-01,0.422770,0.176756,0.126813,-0.380187,-0.346152
...,...,...,...,...,...
2026-02-01,0.026573,0.040612,0.042498,-0.179624,0.069942
2026-03-01,0.118239,0.075455,0.044644,-0.271824,0.033486
2026-04-01,0.102525,0.081651,0.077853,-0.269193,0.007165


In [226]:
t_eval = weights_rolling.index
#np.save('data/t_eval.npy', t_eval)

for name, signal in signals_cleaned.items():
    signals_cleaned[name] = signal.reindex(t_eval)

signal_matrix = np.stack(
    [
        signals_cleaned["beta"].values,   
        signals_cleaned["momentum"].values,     
        signals_cleaned["size"].values,    
        signals_cleaned["low_vol"].values,   
        signals_cleaned["ltr"].values  
    ],
    axis=2   # stack along the 3rd dimension
)

signal_matrix.shape

(102, 60, 5)

In [227]:
weights = weights_rolling.to_numpy()
scores = np.sum(signal_matrix * weights[:, None, :], axis = 2)
scores = pd.DataFrame(
    scores, 
    index = t_eval, 
    columns = all_tickers
)

#scores.to_csv('data/factor_scores.csv')
scores

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,0.308496,0.188353,2.730963,0.564434,0.465089,0.299513,0.545250,0.291479,0.475048,0.341299,...,-0.249550,-0.031149,0.066642,-0.498244,-0.566722,-0.172946,-0.257145,-0.290178,1.696090,0.381379
2018-02-01,-0.028970,0.055925,2.350189,0.291677,0.255501,0.162623,0.301102,0.075980,0.261190,0.265316,...,-0.088615,-0.018668,0.313302,-0.429510,-0.582536,-0.055119,-0.178352,-0.138977,1.052558,0.272717
2018-03-01,-0.167858,0.218679,2.325695,0.075135,0.005538,0.213251,0.423666,-0.001692,0.188352,0.132173,...,0.010869,-0.094877,0.432175,-0.440638,-0.505726,-0.142713,-0.304873,-0.224147,0.546368,0.101062
2018-04-01,0.010111,0.265951,2.463600,0.053080,-0.011400,0.191497,0.414653,-0.044797,0.166730,0.113664,...,-0.159373,-0.111603,0.238215,-0.543229,-0.678089,-0.183379,-0.435572,-0.218073,0.877020,0.068044
2018-05-01,0.250000,0.167516,2.155427,0.226291,-0.098244,0.085943,0.373630,-0.166559,0.075543,0.002737,...,-0.133546,-0.130872,0.212071,-0.637988,-0.778256,-0.273080,-0.580515,-0.390572,1.083000,0.061338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-01,-0.029996,0.096672,0.143444,0.321580,0.111367,-0.183555,-0.057961,0.077790,0.017935,-0.192228,...,-0.064725,-0.155515,0.078653,-0.120699,-0.147502,-0.105530,-0.073726,0.111558,0.281061,-0.201174
2026-03-01,-0.166709,0.121681,0.427691,0.459032,0.213145,-0.305367,-0.141923,0.034488,-0.077473,-0.080846,...,-0.080397,-0.268739,0.039847,-0.163714,-0.087917,-0.204808,-0.124151,0.498879,0.583684,-0.063309
2026-04-01,-0.160959,0.086261,0.417594,0.698778,0.122328,-0.275407,-0.184997,0.010658,-0.027737,-0.090745,...,-0.212518,-0.243735,0.091145,-0.157056,-0.023362,-0.184768,-0.177275,0.547481,0.513523,-0.117069


## 4a. PCA Portfolio

In [228]:
returns = np.log(data).diff().dropna()
returns_eval = returns.shift(-1).reindex(t_eval)
#returns_eval.to_csv('data/returns_eval.csv')
returns_eval.tail()

Ticker,AAPL,ABBV,AEP,AMT,AMZN,APD,AVGO,BAC,BLK,CAT,...,SLB,SO,SPG,T,TSLA,UNH,UNP,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-01,-0.040124,-0.064929,-0.020688,-0.105905,-0.008272,0.052365,-0.029828,-0.016275,-0.094802,-0.047374,...,0.000973,-0.008871,-0.077318,0.034389,-0.079498,-0.072745,-0.088158,-0.022850,-0.027052,0.106632
2026-03-01,0.066902,-0.020517,0.044976,0.066981,0.241121,0.038630,0.299126,0.092228,0.102580,0.230153,...,0.101500,0.001863,0.088105,-0.093455,0.026230,0.314195,0.104998,0.032381,0.059734,-0.094703
2026-04-01,0.140707,0.029836,-0.071962,0.022995,0.020833,-0.074100,0.067927,-0.035412,-0.017724,-0.016127,...,-0.041826,-0.041036,0.005873,-0.052240,0.132709,0.026187,-0.020528,-0.053025,-0.128932,-0.053803
2026-05-01,-0.075524,0.144790,0.077008,-0.124153,-0.127091,0.050938,-0.166230,0.099179,-0.085013,0.195419,...,-0.159880,0.038991,0.098422,-0.180710,-0.035478,0.094504,0.035018,0.063700,-0.021747,-0.060590
2026-06-01,0.079538,-0.021571,-0.014690,0.019644,0.027417,0.028288,0.054754,0.047302,0.077892,-0.115138,...,0.026637,-0.000523,-0.032538,0.020087,-0.033779,0.029725,0.058338,0.053825,0.004317,0.007696


In [229]:
from sklearn.decomposition import PCA

lookback = 36
window = returns.iloc[-lookback-1:-1]
pca = PCA()
pca.fit(window)
variance_pct = pca.explained_variance_ratio_
cum_var = np.cumsum(variance_pct)
k = np.argmax(cum_var >= 0.8) + 1

In [230]:
len(range(lookback, len(returns)))

126

In [231]:
len(returns)

162

In [232]:
threshold = 0.8
sigma_list = []

for t in range(lookback, len(returns)):
    # Lookback window
    window = returns.iloc[t-lookback:t]
    #print(returns.index[t])
    # PCA deconstruction
    pca = PCA()
    pca.fit(window)

    cum_var = np.cumsum(pca.explained_variance_ratio_)
    k = np.argmax(cum_var >= threshold)

    B = pca.components_[:k,:]
    F = np.diag(pca.explained_variance_[:k])

    sigma_pca = B.T @ F @ B

    sample_cov = window.cov().values

    D = np.diag(np.diag(sample_cov - sigma_pca))

    sigma_t = sigma_pca + D

    sigma_list.append(sigma_t)

# Stack along axis 2
sigma = np.stack(sigma_list, axis=2)
sigma.shape


(60, 60, 126)

In [233]:
scores_n = normalise(scores)

scores_n

,AAPL,MSFT,NVDA,AVGO,CRM,JPM,BAC,GS,MS,BLK,...,PLD,AMT,EQIX,SPG,PSA,LIN,APD,NEM,FCX,SHW
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01,0.011202,0.006839,0.099165,0.020495,0.016888,0.010876,0.019799,0.010584,0.017250,0.012393,...,-0.009062,-0.001131,0.002420,-0.018092,-0.020579,-0.006280,-0.009337,-0.010537,0.061588,0.013848
2018-02-01,-0.001309,0.002526,0.106169,0.013176,0.011542,0.007346,0.013602,0.003432,0.011799,0.011986,...,-0.004003,-0.000843,0.014153,-0.019403,-0.026316,-0.002490,-0.008057,-0.006278,0.047549,0.012320
2018-03-01,-0.008328,0.010850,0.115390,0.003728,0.000275,0.010581,0.021020,-0.000084,0.009345,0.006558,...,0.000539,-0.004707,0.021443,-0.021862,-0.025092,-0.007081,-0.015126,-0.011121,0.027108,0.005014
2018-04-01,0.000446,0.011742,0.108772,0.002344,-0.000503,0.008455,0.018308,-0.001978,0.007361,0.005018,...,-0.007037,-0.004927,0.010518,-0.023984,-0.029939,-0.008096,-0.019231,-0.009628,0.038722,0.003004
2018-05-01,0.009654,0.006469,0.083234,0.008738,-0.003794,0.003319,0.014428,-0.006432,0.002917,0.000106,...,-0.005157,-0.005054,0.008189,-0.024637,-0.030053,-0.010545,-0.022417,-0.015082,0.041821,0.002369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-01,-0.003517,0.011335,0.016818,0.037704,0.013057,-0.021521,-0.006796,0.009121,0.002103,-0.022538,...,-0.007589,-0.018234,0.009222,-0.014152,-0.017294,-0.012373,-0.008644,0.013080,0.032953,-0.023587
2026-03-01,-0.011814,0.008623,0.030308,0.032529,0.015104,-0.021640,-0.010057,0.002444,-0.005490,-0.005729,...,-0.005697,-0.019044,0.002824,-0.011602,-0.006230,-0.014514,-0.008798,0.035353,0.041362,-0.004486
2026-04-01,-0.012105,0.006488,0.031407,0.052554,0.009200,-0.020713,-0.013913,0.000802,-0.002086,-0.006825,...,-0.015983,-0.018331,0.006855,-0.011812,-0.001757,-0.013896,-0.013333,0.041175,0.038621,-0.008805


In [234]:
print(t_eval.shape)
sigma_t = sigma[:,:,-len(t_eval):]
sigma_t.shape

(102,)


(60, 60, 102)

In [235]:
weights_t = pd.DataFrame(
    index = t_eval, 
    columns = scores.columns
)
for i,t in enumerate(t_eval):
    mu_t = scores_n.loc[t].values
    sigma_t_i = sigma[:,:,i]

    w = np.linalg.solve(sigma_t_i, mu_t)
    w = w - w.mean() # denorm
    w = w / np.abs(w).sum() # scale exposure

    weights_t.loc[t] = w

print(weights_t.sum(axis = 1))
print(weights_t.abs().sum(axis = 1))

Date
2018-01-01   -0.0
2018-02-01    0.0
2018-03-01   -0.0
2018-04-01   -0.0
2018-05-01   -0.0
             ... 
2026-02-01   -0.0
2026-03-01    0.0
2026-04-01   -0.0
2026-05-01   -0.0
2026-06-01    0.0
Freq: MS, Length: 102, dtype: object
Date
2018-01-01    1.0
2018-02-01    1.0
2018-03-01    1.0
2018-04-01    1.0
2018-05-01    1.0
             ... 
2026-02-01    1.0
2026-03-01    1.0
2026-04-01    1.0
2026-05-01    1.0
2026-06-01    1.0
Freq: MS, Length: 102, dtype: object


In [240]:
weights_t.iloc[-1].sort_values(ascending = False)

GOOGL    0.075625
AMZN     0.072275
NVDA     0.045544
GE       0.045251
AVGO     0.033313
DE       0.030066
NEM      0.027379
UNH      0.026888
CAT      0.024468
TSLA     0.021523
SLB      0.019699
META     0.019023
NFLX     0.017826
FCX      0.015755
CRM      0.014171
LLY       0.00759
MSFT     0.002913
SHW      0.000693
MS      -0.000181
HD      -0.000209
LOW     -0.001038
GS      -0.001973
PSA     -0.002519
AAPL    -0.003138
SPG     -0.003191
COP     -0.003594
BLK     -0.003614
HON     -0.003803
BAC     -0.004088
XOM     -0.004633
EQIX    -0.004876
CVX     -0.006015
T       -0.006852
UNP     -0.006886
MRK     -0.007405
DIS     -0.007479
EOG     -0.007564
CL      -0.008146
JPM     -0.009563
WMT     -0.009637
PFE     -0.010181
AEP       -0.0103
APD     -0.010923
NKE     -0.011897
PLD     -0.013153
JNJ     -0.016811
ABBV    -0.016897
LIN     -0.017711
KO      -0.018364
COST    -0.018607
NEE     -0.019515
WFC     -0.019676
PEP     -0.020169
AMT     -0.020297
PG      -0.021549
DUK     -0

## 4b. Long-only Portfolio

In [ ]:
scores.iloc[-1].sort_values(ascending = False).head(10)

['AVGO', 'CAT', 'GOOGL', 'UNH', 'NVDA', 'TSLA', 'NEM', 'AMZN', 'FCX', 'GE']